# 06 — Multi-agent systems

**Definition:** several decision-makers, each with its own prompt and tools, coordinating
on one problem.

**Why bother** — not because "more agents = smarter". Three concrete limits of one agent:

| problem with a single agent | multi-agent fix |
|---|---|
| 30 tools, the LLM picks the wrong one | each agent gets 3–5 tools |
| one prompt has to cover every domain | one focused prompt per agent |
| everything runs sequentially | independent agents run in parallel |
| one team owns the whole prompt | teams own separate agents |

The cost is real: more LLM calls, more latency, more ways to fail, much harder debugging.
**Don't reach for this until a single ReAct agent has actually failed you.**

## The four topologies

```
(1) SUPERVISOR                   (2) NETWORK (swarm)
   +----------+                     A <--> B
   |supervisor|                     ^ \   / ^
   +----------+                     |  X  |
    /    |    \                     v /   \ v
   A     B     C                    C <--> D
  (all report back)              (anyone -> anyone)

(3) HIERARCHICAL                 (4) PIPELINE
     +-----+                       A -> B -> C -> END
     | top |                      (fixed order, no
     +-----+                       routing decisions)
      /    \
  +----+  +----+
  |sup1|  |sup2|
  +----+  +----+
   /  \    /  \
  A    B  C    D
```

| topology | control | use when | risk |
|---|---|---|---|
| supervisor | central | the default choice | supervisor is a bottleneck |
| network | none | agents are genuine peers | chaos, loops |
| hierarchical | layered | more than ~6 agents | latency stacks up |
| pipeline | fixed | the order is known | it's just a chain |

I build three below: **supervisor**, **`Command` handoffs**, and **`Send` fan-out**.

## Setup

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

# Both providers serve the SAME model (gpt-oss-120b), so behaviour is identical.
# Groq is the default; set LLM_PROVIDER=cerebras to switch. I added that second path
# after burning through Groq's 200k-tokens-per-day cap while writing these notebooks.
if os.environ.get("LLM_PROVIDER", "groq") == "cerebras":
    from langchain_openai import ChatOpenAI

    if not os.environ.get("CEREBRAS_API_KEY"):
        os.environ["CEREBRAS_API_KEY"] = getpass.getpass("CEREBRAS_API_KEY: ")
    llm = ChatOpenAI(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://api.cerebras.ai/v1",
        api_key=os.environ["CEREBRAS_API_KEY"],
        timeout=120,      # ChatOpenAI defaults to NO timeout - a stalled
        max_retries=5,    # connection hangs the whole notebook forever
    )
else:
    from langchain_groq import ChatGroq

    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
    # reasoning_format="hidden" keeps gpt-oss's chain-of-thought out of .content
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        reasoning_format="hidden",
        timeout=120,
        max_retries=5,
    )


def show(graph):
    print(graph.get_graph().draw_mermaid())


print(llm.invoke("Reply with the single word: ready").content)

# Part A — supervisor

## Specialist agents

Each agent gets a **narrow prompt** and **few tools**. That focus is the entire point — give
one agent every tool and you've just built a slower single agent.

Note the writer has **no tools at all**, deliberately. It only composes.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def web_search(query: str) -> str:
    """Search the web for factual information."""
    return (
        f"[results for '{query}'] Kafka handles ~1M msgs/sec per broker; "
        "LinkedIn processes 7 trillion msgs/day; p99 latency ~5ms."
    )


@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. '7e12 / 86400'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


research_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="You are a researcher. Find facts using web_search. Report the findings "
    "only - do not analyse or write prose. Keep it under 80 words.",
)

analyst_agent = create_agent(
    model=llm,
    tools=[calculator],
    system_prompt="You are a data analyst. Do calculations on the facts already gathered "
    "using the calculator tool. Report the numbers only. Keep it under 80 words.",
)

writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a technical writer. Turn the gathered facts and numbers into a "
    "clear 2-paragraph summary. Do not invent new facts. You have no tools.",
)
print("3 specialists ready")

Note I pass `model=llm`, **not** `model=llm.bind_tools([...])`. `create_agent` binds the
`tools=` list itself; pre-binding as well means the schemas get attached twice and the two
lists can silently disagree.

## State + the supervisor's decision schema

In [ ]:
from typing import Annotated, List, Literal

from langgraph.graph import MessagesState
from pydantic import BaseModel, Field

MEMBERS = ["researcher", "analyst", "writer"]


class Router(BaseModel):
    """The supervisor's decision."""

    next: Literal["researcher", "analyst", "writer", "FINISH"] = Field(
        description="Which worker acts next. FINISH when the task is complete."
    )
    reason: str = Field(description="One short sentence explaining the choice.")


class TeamState(MessagesState):
    """Shared state - every agent reads and writes the same `messages` list."""

    next: str                                            # no reducer -> replaced each turn
    completed: Annotated[List[str], lambda a, b: a + b]  # accumulates -> the loop guard

`Literal` on `next` is doing real work: it makes a hallucinated agent name structurally
impossible, which kills a whole class of routing bugs before it starts.

## The supervisor node

Loop prevention lives here. The trick is feeding `completed` back into the prompt — without
it the supervisor has no idea who already ran and will happily ping-pong forever.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

SUPERVISOR_PROMPT = (
    "You are a supervisor managing these workers: {members}.\n"
    "Given the conversation, decide who acts NEXT.\n\n"
    "Typical order: researcher (gather facts) -> analyst (compute) -> writer (compose).\n"
    "Workers already used: {completed}\n\n"
    "Do NOT re-invoke a worker that has already done its job.\n"
    "Respond FINISH once the writer has produced the summary."
)

supervisor_llm = llm.with_structured_output(Router, method="json_schema")


def supervisor_node(state: TeamState) -> dict:
    done = state.get("completed", [])
    system = SUPERVISOR_PROMPT.format(
        members=", ".join(MEMBERS), completed=", ".join(done) or "none"
    )

    decision = supervisor_llm.invoke(
        [SystemMessage(content=system)] + state["messages"]
    )
    print(f"\nSUPERVISOR -> {decision.next}  ({decision.reason})")
    return {"next": decision.next}

### Wrapping a sub-agent as a node

Two reasons this wrapper exists:

1. the sub-agent returns its **own** full message list; I only want its final answer
2. relabelling it as an `AIMessage` with a `name` lets the supervisor tell who said what

Returning the sub-agent's whole history instead would blow up the parent's context — every
worker's internal tool chatter would end up in every later worker's prompt.

In [ ]:
def make_worker(agent, name: str):
    """Factory: wrap a ReAct sub-agent as a single node in the parent graph."""

    def node(state: TeamState) -> dict:
        print(f"  {name} working...")
        result = agent.invoke({"messages": state["messages"]})
        final = result["messages"][-1].content
        print(f"    -> {final[:110]}")

        return {
            "messages": [AIMessage(content=final, name=name)],   # ONLY the last message
            "completed": [name],                                 # reducer appends
        }

    return node


researcher_node = make_worker(research_agent, "researcher")
analyst_node = make_worker(analyst_agent, "analyst")
writer_node = make_worker(writer_agent, "writer")

## Build the star

In [ ]:
from langgraph.graph import END, START, StateGraph

b = StateGraph(TeamState)

b.add_node("supervisor", supervisor_node)
b.add_node("researcher", researcher_node)
b.add_node("analyst", analyst_node)
b.add_node("writer", writer_node)

b.add_edge(START, "supervisor")

for m in MEMBERS:
    b.add_edge(m, "supervisor")     # every worker reports BACK - that's the star

b.add_conditional_edges(
    "supervisor",
    lambda s: s["next"],
    {"researcher": "researcher", "analyst": "analyst", "writer": "writer", "FINISH": END},
)

supervisor_graph = b.compile()
show(supervisor_graph)

In [ ]:
task = (
    "Research Kafka's throughput at scale, work out the messages per second "
    "from the daily figure, then write a short summary for an architect."
)

out = supervisor_graph.invoke(
    {"messages": [HumanMessage(content=task)], "completed": []},
    config={"recursion_limit": 20},      # non-negotiable loop guard
)

print("\n" + "=" * 60)
print(out["messages"][-1].content)
print(f"\nworkers used: {out['completed']}")

### The prebuilt shortcut

`langgraph-supervisor` does all of the above. Agents need a `name` so the supervisor can
address them — that's the one extra requirement versus what I built by hand.

In [ ]:
from langgraph_supervisor import create_supervisor

named_research = create_agent(
    model=llm, tools=[web_search], system_prompt="You are a researcher. Report facts only.",
    name="researcher",
)
named_writer = create_agent(
    model=llm, tools=[], system_prompt="You are a technical writer. One short paragraph.",
    name="writer",
)

prebuilt = create_supervisor(
    agents=[named_research, named_writer],
    model=llm,
    prompt="You manage a researcher and a writer. Delegate to one at a time, then FINISH.",
).compile()

r = prebuilt.invoke(
    {"messages": [HumanMessage("What is Kafka's throughput? Then summarise it in one paragraph.")]},
    config={"recursion_limit": 15},
)
print(r["messages"][-1].content[:600])

# Part B — handoffs with `Command`

No supervisor. Agents hand off **directly**. Cheaper (no extra routing LLM call) but much
easier to get stuck in a loop.

`Command` vs a conditional edge:

| | conditional edge | `Command` |
|---|---|---|
| returns | just the next node name | **state update + next node together** |
| lives in | a separate router function | **inside** the node |
| use for | pure routing | handoffs (update and jump as one move) |

The `Command[Literal[...]]` return annotation is **required** — it's the only way LangGraph
knows where this node can jump, so without it the graph can't be drawn or validated.

In [ ]:
from langgraph.types import Command


def triage(state: MessagesState) -> Command[Literal["billing_bot", "tech_bot", "__end__"]]:
    """Command = state update + jump, in ONE atomic return."""
    text = state["messages"][-1].content.lower()

    if any(w in text for w in ["invoice", "charge", "payment", "refund"]):
        return Command(
            goto="billing_bot",
            update={"messages": [AIMessage(content="Handing off to billing.", name="triage")]},
        )
    if any(w in text for w in ["error", "crash", "bug", "broken"]):
        return Command(
            goto="tech_bot",
            update={"messages": [AIMessage(content="Handing off to tech.", name="triage")]},
        )
    return Command(
        goto="__end__",     # the string form of END
        update={"messages": [AIMessage(content="Could you clarify your issue?", name="triage")]},
    )


def billing_bot(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="Billing: refunded within 5 business days.", name="billing_bot")]}


def tech_bot(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="Tech: please send us your error logs.", name="tech_bot")]}


hb = StateGraph(MessagesState)
hb.add_node("triage", triage)
hb.add_node("billing_bot", billing_bot)
hb.add_node("tech_bot", tech_bot)

hb.add_edge(START, "triage")
# NO conditional edges needed - Command carries the routing itself.
hb.add_edge("billing_bot", END)
hb.add_edge("tech_bot", END)

handoff_graph = hb.compile()

for q in ["I was charged twice", "The app crashes on launch", "Do you have a newsletter?"]:
    r = handoff_graph.invoke({"messages": [HumanMessage(content=q)]})
    last = r["messages"][-1]
    print(f"{q!r:32} -> {last.name}: {last.content}")

This triage is deterministic `if/else` on purpose, so the routing mechanism is visible
without an LLM in the way. The real swarm pattern gives each agent a `transfer_to_X` **tool**
so the *model* decides the handoff:

```python
@tool(f"transfer_to_{agent_name}", description=f"Transfer to the {agent_name} agent.")
def handoff(
    state: Annotated[dict, InjectedState],              # injected, LLM never sees it
    tool_call_id: Annotated[str, InjectedToolCallId],   # injected, LLM never sees it
) -> Command:
    # MUST return a ToolMessage or the provider rejects the unanswered tool_call
    msg = ToolMessage(content=f"Transferred to {agent_name}", tool_call_id=tool_call_id)
    return Command(
        goto=agent_name,
        update={"messages": state["messages"] + [msg]},
        graph=Command.PARENT,      # jump in the PARENT graph, not this subgraph
    )
```

Two things to remember: `InjectedState` / `InjectedToolCallId` are hidden from the schema
the LLM sees, and `Command.PARENT` is what lets a tool inside a sub-agent move the outer graph.

# Part C — parallel fan-out with `Send`

The only topology that gives a **real speedup**: all branches run concurrently.

`Send(node_name, state)` means "run `node_name` with **this** state, now, in parallel".
Two rules that trip everyone up:

1. each `Send` gets its **own private state** — not the parent's
2. results **must** merge through a reducer, or they overwrite each other and you silently
   keep only one

In [ ]:
import operator
from typing import TypedDict

from langgraph.types import Send


class MapState(TypedDict):
    topic: str
    subtopics: List[str]
    # WITHOUT operator.add the parallel workers OVERWRITE each other. Mandatory.
    summaries: Annotated[List[str], operator.add]
    final: str


class WorkerState(TypedDict):
    """Each Send gets its OWN state - only the keys I put in the Send are visible."""

    subtopic: str


class Subtopics(BaseModel):
    subtopics: List[str] = Field(description="3 distinct, independent subtopics")


def split(state: MapState) -> dict:
    """MAP phase: break the topic into independent chunks."""
    r = llm.with_structured_output(Subtopics, method="json_schema").invoke(
        f"Break '{state['topic']}' into exactly 3 independent subtopics."
    )
    print(f"split into: {r.subtopics}")
    return {"subtopics": r.subtopics}


def fan_out(state: MapState):
    """Returns a LIST of Sends -> N parallel copies of `research_one`.
    Not a normal router: it returns Sends, not node names."""
    return [Send("research_one", {"subtopic": s}) for s in state["subtopics"]]


def research_one(state: WorkerState) -> dict:
    """Runs in parallel. Sees ONLY {"subtopic": ...} - nothing else from the parent."""
    text = llm.invoke(f"Write 2 concise sentences about: {state['subtopic']}").content
    print(f"  done: {state['subtopic']}")
    return {"summaries": [f"**{state['subtopic']}**: {text}"]}


def reduce_node(state: MapState) -> dict:
    """REDUCE phase: all parallel results have merged into state['summaries']."""
    joined = "\n\n".join(state["summaries"])
    return {"final": llm.invoke(f"Combine into one coherent overview:\n\n{joined}").content}

In [ ]:
mb = StateGraph(MapState)
mb.add_node("split", split)
mb.add_node("research_one", research_one)
mb.add_node("reduce", reduce_node)

mb.add_edge(START, "split")
mb.add_conditional_edges("split", fan_out, ["research_one"])   # the fan-out edge
mb.add_edge("research_one", "reduce")    # converges once ALL branches finish
mb.add_edge("reduce", END)

map_graph = mb.compile()

res = map_graph.invoke({"topic": "Event-driven architecture with Kafka", "summaries": []})
print(f"\ncollected {len(res['summaries'])} summaries in parallel\n")
print(res["final"][:700])

### Proving the reducer is load-bearing

Same graph, one change: `summaries` is a plain `List[str]` with no reducer. I expected the
classic silent-data-loss story — three workers run, one result survives, no warning. Let's
see what actually happens (no LLM calls here, the worker just echoes its subtopic).

In [ ]:
class BrokenMapState(TypedDict):
    topic: str
    subtopics: List[str]
    summaries: List[str]      # <- no reducer
    final: str


def cheap_worker(state: WorkerState) -> dict:
    return {"summaries": [state["subtopic"]]}


xb = StateGraph(BrokenMapState)
xb.add_node("split", lambda s: {"subtopics": ["alpha", "beta", "gamma"]})
xb.add_node("research_one", cheap_worker)
xb.add_edge(START, "split")
xb.add_conditional_edges(
    "split", lambda s: [Send("research_one", {"subtopic": t}) for t in s["subtopics"]], ["research_one"]
)
xb.add_edge("research_one", END)

try:
    print("no reducer ->", xb.compile().invoke({"topic": "x", "summaries": []})["summaries"])
except Exception as e:
    print(f"no reducer RAISED {type(e).__name__}:\n  {e}")

Good news, and worth correcting my own mental model: LangGraph **refuses** the write rather
than silently keeping one value.

```
InvalidUpdateError: At key 'summaries': Can receive only one value per step.
Use an Annotated key to handle multiple values.
```

Two branches writing the same un-reduced key in the same superstep is a hard error. The
old "parallel results vanish quietly" folklore predates this check — the failure is loud
now. It still only fires when two branches land in the *same* step, so a fan-out that
happens to run one branch at a time won't surface it.

## Notes to self

**Do sub-agents share state or get their own?** The central design decision.

| | shared state | isolated (subgraph) |
|---|---|---|
| sub-agent sees | everything | only what I pass in |
| coupling | tight | loose |
| context cost | grows fast | controlled |
| use when | agents need each other's work | agents are independent specialists |

Default to a shared `messages` list for simplicity; isolate when the context blows up.

**Is a supervisor just a router?** Almost — but a router runs once, while a supervisor runs
**after every worker** and re-decides. That loop is what makes it agentic rather than a
switch statement.

**Loop prevention checklist** (I need most of these, not one):

- [ ] `Literal` restricts the supervisor's options
- [ ] `completed` fed back into the supervisor prompt
- [ ] `FINISH` is an explicit, well-described option
- [ ] `recursion_limit` set
- [ ] only the worker's **last** message returns to the parent

**Failure modes:**

| symptom | cause | fix |
|---|---|---|
| supervisor ping-pongs | no memory of who ran | track `completed`, feed it into the prompt |
| context explodes | workers return full history | return only `messages[-1]` |
| `InvalidUpdateError` on fan-out | no reducer on the merge key | `Annotated[list, operator.add]` |
| hallucinated agent name | free-text routing | `Literal[...]` in the Router schema |
| `Command` node not drawn | missing return annotation | `-> Command[Literal["x"]]` |
| provider 400 on handoff | a `tool_call` never answered | return a `ToolMessage` with the id |
| slower than one agent | sequential supervisor | use `Send` for independent work |

**API I used:**

```python
Command(goto=..., update=...)           # atomic update + jump
Command[Literal["a", "b"]]              # REQUIRED return annotation
Command.PARENT                          # jump in the outer graph
Send("node", {...})                     # parallel branch with private state
Annotated[list, operator.add]           # MANDATORY for parallel merges
Literal[...] on the Router schema       # no hallucinated agent names
create_supervisor(agents=[...], model=llm, prompt=...)
```

Next: **07 — Autonomous**, where the agent owns the loop and decides when it's done.